In [0]:
stream_df = (
                    spark.readStream
                         .format("cloudFiles")
                         .option("cloudFiles.format", "json")
                         .option("cloudFiles.schemaLocation", "/Volumes/traffic_project/raw/stream/schema_v2/")
                         .option("cloudFiles.inferColumnTypes", "true")
                         .option("cloudFiles.schemaHints",   "event_time TIMESTAMP, detection_confidence DOUBLE")
                         .option("pathGlobFilter", "batch_*.json")
                         .load("/Volumes/traffic_project/raw/stream/landing/")
)

In [0]:
from pyspark.sql.functions import current_timestamp, col

transformed_df = (
                                stream_df.withColumn("file_path", col("_metadata.file_path"))
                                            .withColumn("ingestion_date", current_timestamp())
)

In [0]:
streaming_query = (
                    transformed_df.writeStream
                        .format("delta")
                        .trigger(availableNow=True)
                        .option("checkpointLocation", "/Volumes/traffic_project/raw/stream/_checkpoint_v2/")
                        .toTable("traffic_project.bronze.vehicles")
)